# Chapter 8: Online DPO & Iterative Alignment

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arunpshankar/packt-final/blob/main/code/notebooks/part2_core/08_online_dpo.ipynb)

> **Book:** *Reinforcement Learning for Large Language Models* — Arun Shankar (Packt, 2025)  
> **Chapter 8:** Online DPO & Iterative Alignment  
> **Notebook:** `part2_core/08_online_dpo.ipynb`

---

## What this notebook covers

This is the companion notebook for Chapter 8 of the book. Run it on a free Colab T4 GPU. All code uses small, publicly available models (under 500 MB) that fit within the free tier memory limit.

**To open in Colab:** Click the badge above, or replace `github.com` with `githubtocolab.com` in the URL.


In [ ]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    %pip install -q transformers==4.40.0 trl==0.8.6 peft==0.10.0 \
        datasets==2.19.0 accelerate==0.29.3

print('Environment ready.')

In [ ]:
import warnings
import random
import copy
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModelForCausalLM

warnings.filterwarnings('ignore')

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU : {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Part 1 — DPO Loss from Scratch

Before building the online loop we implement the DPO loss in pure PyTorch so the maths is transparent.

### What the formula says

DPO works by implicitly treating the log-ratio

$$\hat{r}_\theta(x, y) = \beta\log\frac{\pi_\theta(y|x)}{\pi_{\text{ref}}(y|x)}$$

as a reward. It then pushes $\hat{r}(y_w) - \hat{r}(y_l)$ to be as large and positive as possible, which is exactly what the Bradley-Terry model says the reward should do.

### The 20-line implementation

In [ ]:
def sequence_log_prob(
    model: torch.nn.Module,
    input_ids: torch.Tensor,
    attention_mask: torch.Tensor,
    response_start: int,
) -> torch.Tensor:
    """
    Compute sum of log-probabilities over the *response* tokens only.

    Parameters
    ----------
    model         : causal LM
    input_ids     : (1, T)
    attention_mask: (1, T)
    response_start: token index where the response begins

    Returns
    -------
    log_prob : scalar tensor
    """
    with torch.no_grad():
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    # shift: logits[t] predicts token[t+1]
    log_probs = F.log_softmax(logits[:, :-1, :], dim=-1)  # (1, T-1, V)
    target_ids = input_ids[:, 1:]                          # (1, T-1)
    # gather log-prob of each actual token
    token_log_p = log_probs.gather(
        dim=2, index=target_ids.unsqueeze(-1)
    ).squeeze(-1)                                          # (1, T-1)
    # sum only over response tokens (zero-pad prompt positions)
    resp_mask = torch.zeros_like(token_log_p)
    resp_mask[:, response_start - 1:] = 1.0               # -1 because of shift
    return (token_log_p * resp_mask).sum(dim=1)            # (1,)


def dpo_loss(
    policy_model: torch.nn.Module,
    ref_model: torch.nn.Module,
    chosen_ids: torch.Tensor,
    chosen_mask: torch.Tensor,
    rejected_ids: torch.Tensor,
    rejected_mask: torch.Tensor,
    response_start_chosen: int,
    response_start_rejected: int,
    beta: float = 0.1,
) -> torch.Tensor:
    """
    DPO loss (negative log-sigmoid of the margin).

    L = -log sigmoid( beta * (log pi_theta(y_w|x) - log pi_ref(y_w|x))
                    - beta * (log pi_theta(y_l|x) - log pi_ref(y_l|x)) )
    """
    # Policy log-probs
    pi_logp_w = sequence_log_prob(policy_model, chosen_ids,   chosen_mask,   response_start_chosen)
    pi_logp_l = sequence_log_prob(policy_model, rejected_ids, rejected_mask, response_start_rejected)

    # Reference log-probs (no grad needed)
    with torch.no_grad():
        ref_logp_w = sequence_log_prob(ref_model, chosen_ids,   chosen_mask,   response_start_chosen)
        ref_logp_l = sequence_log_prob(ref_model, rejected_ids, rejected_mask, response_start_rejected)

    # Implicit reward margin
    chosen_reward   = beta * (pi_logp_w - ref_logp_w)
    rejected_reward = beta * (pi_logp_l - ref_logp_l)
    margin = chosen_reward - rejected_reward

    return -F.logsigmoid(margin).mean()


print('DPO loss function defined.')

In [ ]:
# Quick smoke-test: margin should shrink (loss decreases) with gradient descent
import torch.optim as optim

toy_model = AutoModelForCausalLM.from_pretrained('distilgpt2').to(DEVICE)
toy_ref   = copy.deepcopy(toy_model)  # frozen reference
toy_ref.eval()

tok = AutoTokenizer.from_pretrained('distilgpt2')
tok.pad_token = tok.eos_token

prompt   = "### Instruction:\nWhat is 2+2?\n\n### Response:\n"
chosen   = prompt + "2 plus 2 equals 4."
rejected = prompt + "I don't know."

enc_c = tok(chosen,   return_tensors='pt', padding='max_length', max_length=40, truncation=True)
enc_r = tok(rejected, return_tensors='pt', padding='max_length', max_length=40, truncation=True)
prompt_len = len(tok.encode(prompt))

enc_c = {k: v.to(DEVICE) for k, v in enc_c.items()}
enc_r = {k: v.to(DEVICE) for k, v in enc_r.items()}

opt = optim.AdamW(toy_model.parameters(), lr=5e-5)

print('Smoke test — 5 DPO steps:')
for s in range(5):
    toy_model.train()
    loss = dpo_loss(
        toy_model, toy_ref,
        enc_c['input_ids'], enc_c['attention_mask'],
        enc_r['input_ids'], enc_r['attention_mask'],
        prompt_len, prompt_len,
    )
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(toy_model.parameters(), 1.0)
    opt.step()
    print(f'  step {s+1}  loss={loss.item():.4f}')

del toy_model, toy_ref  # free memory
torch.cuda.empty_cache() if DEVICE == 'cuda' else None

## Part 2 — Static DPO on a Fixed Dataset (Baseline)

Before implementing Online DPO we train a **static DPO** baseline on a fixed preference dataset. This is the vanilla DPO setting from Chapter 7 — training data is never refreshed.

We will use this as the comparison point for the online variant.

In [ ]:
STATIC_PAIRS = [
    {"prompt": "What is the capital of France?",
     "chosen":   "The capital of France is Paris.",
     "rejected": "France has cities."},
    {"prompt": "What is 5 + 3?",
     "chosen":   "5 plus 3 equals 8.",
     "rejected": "Numbers are hard."},
    {"prompt": "Who wrote Hamlet?",
     "chosen":   "Hamlet was written by William Shakespeare.",
     "rejected": "Some playwright."},
    {"prompt": "What is the boiling point of water?",
     "chosen":   "Water boils at 100 degrees Celsius at standard pressure.",
     "rejected": "Very hot."},
    {"prompt": "Name a primary color.",
     "chosen":   "Red is a primary color.",
     "rejected": "Colors exist."},
    {"prompt": "What does CPU stand for?",
     "chosen":   "CPU stands for Central Processing Unit.",
     "rejected": "Computer part."},
    {"prompt": "What is the speed of light?",
     "chosen":   "The speed of light is approximately 299,792,458 metres per second.",
     "rejected": "Very fast."},
    {"prompt": "What is machine learning?",
     "chosen":   "Machine learning is a branch of AI where models learn patterns from data.",
     "rejected": "Computers doing stuff."},
    {"prompt": "What planet is closest to the Sun?",
     "chosen":   "Mercury is the closest planet to the Sun.",
     "rejected": "A small one."},
    {"prompt": "What is DNA?",
     "chosen":   "DNA carries genetic information for all living organisms.",
     "rejected": "Biology stuff."},
]

print(f'Static DPO dataset: {len(STATIC_PAIRS)} pairs')

In [ ]:
MODEL_NAME = 'distilgpt2'

static_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
static_tokenizer.pad_token = static_tokenizer.eos_token
static_tokenizer.padding_side = 'right'

static_policy  = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
static_ref     = copy.deepcopy(static_policy)
static_ref.eval()
for p in static_ref.parameters():
    p.requires_grad = False


def encode_pair(prompt_text, response_text, tokenizer, max_len=96):
    full = f"### Instruction:\n{prompt_text}\n\n### Response:\n{response_text}"
    enc  = tokenizer(
        full, return_tensors='pt',
        padding='max_length', max_length=max_len, truncation=True
    )
    prefix = f"### Instruction:\n{prompt_text}\n\n### Response:\n"
    prefix_len = len(tokenizer.encode(prefix))
    return enc['input_ids'].to(DEVICE), enc['attention_mask'].to(DEVICE), prefix_len


static_opt  = optim.AdamW(static_policy.parameters(), lr=5e-5)
STATIC_EPOCHS = 4
static_loss_log = []

print('Training static DPO ...')
for epoch in range(STATIC_EPOCHS):
    random.shuffle(STATIC_PAIRS)
    epoch_losses = []
    static_policy.train()

    for pair in STATIC_PAIRS:
        ids_w, mask_w, rs_w = encode_pair(pair['prompt'], pair['chosen'],   static_tokenizer)
        ids_l, mask_l, rs_l = encode_pair(pair['prompt'], pair['rejected'], static_tokenizer)

        loss = dpo_loss(
            static_policy, static_ref,
            ids_w, mask_w, ids_l, mask_l,
            rs_w, rs_l, beta=0.1
        )
        static_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(static_policy.parameters(), 1.0)
        static_opt.step()
        epoch_losses.append(loss.item())

    avg = float(np.mean(epoch_losses))
    static_loss_log.append(avg)
    print(f'  Epoch {epoch+1}/{STATIC_EPOCHS}  loss={avg:.4f}')

print('Static DPO complete.')

## Part 3 — Online DPO: Self-Generated Preferences

The key idea in Online DPO:

1. **Generate** two responses per prompt using temperature sampling (diversity).
2. **Score** both responses with a rule-based proxy reward (fast, no separate RM).
3. **Assign** `chosen = higher score`, `rejected = lower score`.
4. **Compute** DPO loss on this freshly generated pair.
5. **Update** the policy weights.
6. **Repeat** — each iteration the data comes from the *current* policy.

### Why does this help?
Static DPO trains on data collected from a *historical* model. As the policy improves the static data becomes stale, and the policy keeps trying to contrast responses it no longer produces. Online DPO avoids this by keeping the training distribution on-policy at all times.

In [ ]:
ONLINE_PROMPTS = [
    "What is the capital of Japan?",
    "How many planets are in the solar system?",
    "Who invented the telephone?",
    "What is the largest ocean?",
    "What is 7 times 8?",
    "What does HTML stand for?",
    "What is the freezing point of water?",
    "Name the largest continent.",
    "What is photosynthesis?",
    "What is the powerhouse of the cell?",
    "What is Newton's second law?",
    "Who wrote Macbeth?",
    "What is the square root of 81?",
    "What is the boiling point of water in Fahrenheit?",
    "Explain what gravity is.",
]

print(f'Online DPO prompt pool: {len(ONLINE_PROMPTS)} prompts')

In [ ]:
GOOD_KEYWORDS = [
    'is', 'are', 'the', 'equals', 'stands for', 'was', 'were',
    'degrees', 'metres', 'kilometers', 'named', 'called', 'known',
]

def rule_based_reward(response: str) -> float:
    """
    Proxy reward combining length (log-scaled) and keyword bonus.
    Longer, more informative responses score higher.

    Returns a scalar in roughly [0, 3].
    """
    resp_lower = response.lower()
    words = resp_lower.split()
    length_score  = np.log1p(len(words)) / np.log1p(40)  # normalise to ~1 for 40 words
    keyword_bonus = sum(1 for kw in GOOD_KEYWORDS if kw in resp_lower) * 0.1
    return float(np.clip(length_score + keyword_bonus, 0.0, 3.0))


# Demonstrate the reward function
examples = [
    "Mercury is the planet closest to the Sun.",
    "Small one.",
    "The freezing point of water is 0 degrees Celsius at standard atmospheric pressure.",
]
print('Proxy reward scores:')
for ex in examples:
    print(f'  score={rule_based_reward(ex):.3f}  "{ex}"')

In [ ]:
def generate_two_responses(
    model, tokenizer, prompt: str,
    max_new_tokens: int = 40,
    device: str = 'cpu'
):
    """
    Sample two independent responses for the same prompt.
    Returns (resp_a, resp_b) as decoded strings.
    """
    formatted = f"### Instruction:\n{prompt}\n\n### Response:\n"
    enc = tokenizer(
        formatted, return_tensors='pt',
        truncation=True, max_length=80
    )
    enc = {k: v.to(device) for k, v in enc.items()}
    prompt_len = enc['input_ids'].shape[1]

    model.eval()
    with torch.no_grad():
        out = model.generate(
            **enc,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.9,
            top_k=50,
            num_return_sequences=2,
            pad_token_id=tokenizer.eos_token_id,
        )

    resp_a = tokenizer.decode(out[0][prompt_len:], skip_special_tokens=True).strip()
    resp_b = tokenizer.decode(out[1][prompt_len:], skip_special_tokens=True).strip()
    return resp_a, resp_b


print('generate_two_responses defined.')

In [ ]:
online_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
online_tokenizer.pad_token = online_tokenizer.eos_token
online_tokenizer.padding_side = 'right'

online_policy = AutoModelForCausalLM.from_pretrained(MODEL_NAME).to(DEVICE)
online_ref    = copy.deepcopy(online_policy)
online_ref.eval()
for p in online_ref.parameters():
    p.requires_grad = False

online_opt = optim.AdamW(online_policy.parameters(), lr=5e-5)

ONLINE_STEPS     = 50
online_loss_log  = []
online_score_log = []

print(f'Running Online DPO for {ONLINE_STEPS} steps ...')

for step in range(ONLINE_STEPS):
    prompt = random.choice(ONLINE_PROMPTS)

    # --- 1. Generate two responses from current policy ---
    resp_a, resp_b = generate_two_responses(
        online_policy, online_tokenizer, prompt, device=DEVICE
    )

    # --- 2. Score with proxy reward ---
    score_a = rule_based_reward(resp_a)
    score_b = rule_based_reward(resp_b)

    # --- 3. Assign chosen / rejected ---
    if score_a >= score_b:
        chosen_resp, rejected_resp = resp_a, resp_b
        score_chosen = score_a
    else:
        chosen_resp, rejected_resp = resp_b, resp_a
        score_chosen = score_b

    online_score_log.append(score_chosen)

    # --- 4. Compute DPO loss on the self-generated pair ---
    ids_w, mask_w, rs_w = encode_pair(prompt, chosen_resp,   online_tokenizer)
    ids_l, mask_l, rs_l = encode_pair(prompt, rejected_resp, online_tokenizer)

    online_policy.train()
    loss = dpo_loss(
        online_policy, online_ref,
        ids_w, mask_w, ids_l, mask_l,
        rs_w, rs_l, beta=0.1
    )

    # --- 5. Gradient update ---
    online_opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(online_policy.parameters(), 1.0)
    online_opt.step()

    online_loss_log.append(loss.item())

    if (step + 1) % 10 == 0:
        print(f'Step {step+1:3d}/{ONLINE_STEPS}  '
              f'loss={loss.item():.4f}  score_chosen={score_chosen:.3f}')

print('Online DPO training complete!')

## Part 4 — Visualising Online DPO Training

In [ ]:
def smooth(data, w=7):
    if len(data) < w:
        return np.array(data)
    return np.convolve(data, np.ones(w) / w, mode='valid')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

steps = np.arange(1, ONLINE_STEPS + 1)

# Loss
ax1.plot(steps, online_loss_log, alpha=0.25, color='steelblue')
sl = smooth(online_loss_log)
ax1.plot(np.arange(1, len(sl) + 1), sl, color='steelblue', linewidth=2, label='smoothed')
ax1.set_xlabel('Online DPO Step')
ax1.set_ylabel('DPO Loss')
ax1.set_title('Online DPO — Training Loss')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Proxy score of chosen response
ax2.plot(steps, online_score_log, alpha=0.25, color='orange')
ss = smooth(online_score_log)
ax2.plot(np.arange(1, len(ss) + 1), ss, color='orange', linewidth=2, label='smoothed')
ax2.set_xlabel('Online DPO Step')
ax2.set_ylabel('Proxy Reward (chosen)')
ax2.set_title('Online DPO — Chosen Response Quality')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('online_dpo_training.png', dpi=100, bbox_inches='tight')
plt.show()

## Part 5 — Static DPO vs. Online DPO Evaluation

We evaluate both policies on a held-out set of 10 prompts. For each prompt we generate one response per model and score it with the proxy reward. Higher average score wins.

In [ ]:
EVAL_PROMPTS = [
    "What is the capital of Australia?",
    "What is 9 times 6?",
    "What does RAM stand for?",
    "Who painted the Mona Lisa?",
    "What is the chemical symbol for gold?",
    "How many continents are there?",
    "What is the longest river in the world?",
    "What is the smallest prime number?",
    "What language has the most native speakers?",
    "What is the distance from Earth to the Moon?",
]


def eval_policy(model, tokenizer, prompts, device, max_new_tokens=40):
    """Generate one response per prompt and return proxy reward scores."""
    scores = []
    model.eval()
    for pr in prompts:
        formatted = f"### Instruction:\n{pr}\n\n### Response:\n"
        enc = tokenizer(
            formatted, return_tensors='pt', truncation=True, max_length=80
        )
        enc = {k: v.to(device) for k, v in enc.items()}
        plen = enc['input_ids'].shape[1]
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )
        resp = tokenizer.decode(out[0][plen:], skip_special_tokens=True).strip()
        scores.append(rule_based_reward(resp))
    return scores


static_scores = eval_policy(static_policy, static_tokenizer, EVAL_PROMPTS, DEVICE)
online_scores = eval_policy(online_policy, online_tokenizer, EVAL_PROMPTS, DEVICE)

print(f'Static DPO  avg score: {np.mean(static_scores):.4f}')
print(f'Online DPO  avg score: {np.mean(online_scores):.4f}')

In [ ]:
x = np.arange(len(EVAL_PROMPTS))
fig, ax = plt.subplots(figsize=(11, 4))
ax.bar(x - 0.2, static_scores, 0.4, label=f'Static DPO (mean={np.mean(static_scores):.2f})', color='salmon')
ax.bar(x + 0.2, online_scores, 0.4, label=f'Online DPO (mean={np.mean(online_scores):.2f})', color='steelblue')
ax.set_xticks(x)
ax.set_xticklabels([f'P{i+1}' for i in range(len(EVAL_PROMPTS))])
ax.set_ylabel('Proxy Reward Score')
ax.set_title('Static DPO vs. Online DPO — Eval Proxy Reward')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('static_vs_online_dpo.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Head-to-head wins
online_wins  = sum(o > s for o, s in zip(online_scores, static_scores))
static_wins  = sum(s > o for o, s in zip(online_scores, static_scores))
ties         = len(EVAL_PROMPTS) - online_wins - static_wins
print('Head-to-head on eval prompts:')
print(f'  Online DPO wins : {online_wins}/{len(EVAL_PROMPTS)}')
print(f'  Static DPO wins : {static_wins}/{len(EVAL_PROMPTS)}')
print(f'  Ties            : {ties}/{len(EVAL_PROMPTS)}')

## Part 6 — Visualising the DPO Loss Landscape

One helpful diagnostic is to plot the **implicit reward margin** — the difference in log-ratios between chosen and rejected. A well-trained DPO policy should have a consistently positive margin.

In [ ]:
def compute_margin(policy, ref, tokenizer, pairs, device, beta=0.1):
    """Compute implicit reward margin for a list of preference pairs."""
    margins = []
    policy.eval()
    for pair in pairs:
        ids_w, mask_w, rs_w = encode_pair(pair['prompt'], pair['chosen'],   tokenizer)
        ids_l, mask_l, rs_l = encode_pair(pair['prompt'], pair['rejected'], tokenizer)
        with torch.no_grad():
            pi_w  = sequence_log_prob(policy, ids_w, mask_w, rs_w)
            pi_l  = sequence_log_prob(policy, ids_l, mask_l, rs_l)
            ref_w = sequence_log_prob(ref,    ids_w, mask_w, rs_w)
            ref_l = sequence_log_prob(ref,    ids_l, mask_l, rs_l)
        margin = float(beta * ((pi_w - ref_w) - (pi_l - ref_l)).item())
        margins.append(margin)
    return margins


static_margins = compute_margin(
    static_policy, static_ref, static_tokenizer, STATIC_PAIRS, DEVICE
)
online_margins = compute_margin(
    online_policy, online_ref, online_tokenizer, STATIC_PAIRS, DEVICE
)

fig, ax = plt.subplots(figsize=(10, 4))
xi = np.arange(len(STATIC_PAIRS))
ax.bar(xi - 0.2, static_margins, 0.4, label='Static DPO margins', color='salmon', alpha=0.8)
ax.bar(xi + 0.2, online_margins, 0.4, label='Online DPO margins', color='steelblue', alpha=0.8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Preference Pair Index')
ax.set_ylabel('Implicit Reward Margin β(log π/π_ref)')
ax.set_title('DPO Implicit Reward Margins on Training Pairs')
ax.legend()
ax.grid(True, alpha=0.3, axis='y')
plt.tight_layout()
plt.savefig('dpo_margins.png', dpi=100, bbox_inches='tight')
plt.show()

print(f'Static DPO mean margin: {np.mean(static_margins):+.4f}')
print(f'Online DPO mean margin: {np.mean(online_margins):+.4f}')

## Summary — Online DPO vs. Static DPO

| Aspect | Static DPO | Online DPO |
|--------|-----------|------------|
| Data source | Fixed dataset collected once | Model's own samples at each step |
| Distribution | Shifts away from current policy over time | Always on-policy |
| Reward model | Not required (preferences pre-labelled) | Not required (proxy rule or fast RM) |
| Compute | One pass through dataset | Inference cost at each step |
| Risk | Off-policy degradation | Reward hacking if proxy is weak |

### Key takeaways
- Online DPO keeps training data on-policy, addressing the **distribution shift** problem of static DPO.
- The proxy reward must be chosen carefully — a weak reward leads to reward hacking (Chapter 9).
- Iterative / online variants generalise: the same loop works with real human labels or a learned RM.

> **Next chapter** → Chapter 9 dives into reward model architecture, calibration, and the adversarial failure modes (reward hacking) that the online setting is designed to mitigate.

In [ ]:
print('=== Online DPO — Training Summary ===')
print(f'Static DPO preference pairs : {len(STATIC_PAIRS)}')
print(f'Online DPO steps            : {ONLINE_STEPS}')
print(f'Online DPO prompts pool     : {len(ONLINE_PROMPTS)}')
print(f'Eval prompts                : {len(EVAL_PROMPTS)}')
print(f'Static DPO avg eval score   : {np.mean(static_scores):.4f}')
print(f'Online DPO avg eval score   : {np.mean(online_scores):.4f}')
delta = np.mean(online_scores) - np.mean(static_scores)
print(f'Online improvement          : {"+" if delta >= 0 else ""}{delta:.4f}')
print(f'Static DPO mean margin      : {np.mean(static_margins):+.4f}')
print(f'Online DPO mean margin      : {np.mean(online_margins):+.4f}')